# vLLM probe — free T4

Answers four questions the Keel build cannot answer on its own, none of which
need a cluster and none of which cost anything:

1. **What are vLLM's real Prometheus metric names?** The KEDA trigger in
   `control/render/manifests.py:_scaled_object` queries one that has never been
   observed. If it is wrong, lane C comes up healthy and silently never scales.
2. **Do our catalog engine args work?** `max-model-len: 8192`,
   `gpu-memory-utilization: 0.60`.
3. **What does an OOM actually say?** `control/provisioner/status.py` claims to
   detect it and offers advice; the message should match reality.
4. **Does a chat template apply cleanly?** The provisioner's smoke test assumes
   a real completion comes back.

**Runtime → Change runtime type → T4 GPU** before running anything.

Copy the SUMMARY block at the end back into the repo.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


## 1 · Install

Several minutes. A T4 is compute capability 7.5, so **bfloat16 is not
supported** — anything Ampere or newer would take it. Qwen2.5 declares bf16 in
its config, so this is a real constraint we would hit on any T4 fleet.


In [ ]:
!pip install -q vllm
import vllm; print("vllm", vllm.__version__)


## 2 · Start with our catalog args, unmodified

Exactly what `catalog/qwen2.5-0.5b-instruct.yaml` specifies, plus `--dtype half`
for the T4. Note what vLLM says about dtype during startup — if it warns or
downcasts, that belongs in the catalog as an explicit flag rather than left to
chance.


In [ ]:
import subprocess, time, requests, threading, os, re

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ARGS = [
    "--model", MODEL,
    "--port", "8000",
    "--max-model-len", "8192",          # catalog engine_args
    "--gpu-memory-utilization", "0.60", # catalog engine_args
    "--dtype", "half",                  # T4: no bfloat16
]

log = open("/content/vllm.log", "w")
proc = subprocess.Popen(["vllm", "serve", *ARGS], stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
ready = False
while time.time() - t0 < 900:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            ready = True
            break
    except Exception:
        pass
    if proc.poll() is not None:
        print("SERVER EXITED -- last 40 lines:")
        print("".join(open("/content/vllm.log").readlines()[-40:]))
        break
    time.sleep(5)

LOAD_SECONDS = round(time.time() - t0, 1)
print(f"ready={ready}  load_seconds={LOAD_SECONDS}")


In [ ]:
# What vLLM said about dtype, memory and the KV cache while starting.
import re
txt = open("/content/vllm.log").read()
for line in txt.splitlines():
    if re.search(r"dtype|bfloat16|float16|KV cache|memory|blocks", line, re.I):
        print(line[:200])


## 3 · The metric names — the most valuable output

Everything with a `vllm:` prefix. Compare against what `_scaled_object` queries
(`vllm:request_success_total`) and what `deployment_metrics` expects to feed on
(TTFT, TPOT, queue depth).


In [ ]:
import requests, re
metrics = requests.get("http://localhost:8000/metrics", timeout=10).text
names = sorted({m.group(1) for m in re.finditer(r"^(vllm:[a-zA-Z0-9_]+)", metrics, re.M)})
print(f"{len(names)} vllm: metrics\n")
for n in names:
    print(" ", n)

print("\n--- the ones Keel depends on ---")
for want in ["vllm:request_success_total", "vllm:num_requests_waiting",
             "vllm:num_requests_running", "vllm:time_to_first_token_seconds",
             "vllm:time_per_output_token_seconds", "vllm:gpu_cache_usage_perc"]:
    print(f"  {'FOUND  ' if want in names else 'MISSING'} {want}")


## 4 · The smoke test, exactly as the provisioner runs it

`control/provisioner/main.py` sends this prompt and treats any reply as a pass.
Check the reply is coherent — a chat-template mismatch produces text that is
technically a 200 but obviously wrong.


In [ ]:
import requests, time
t0 = time.time()
r = requests.post("http://localhost:8000/v1/chat/completions", timeout=60, json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Reply with the single word: ok"}],
    "max_tokens": 16,
})
SMOKE = r.json()
print("status", r.status_code, f"in {time.time()-t0:.2f}s")
print("reply:", repr(SMOKE["choices"][0]["message"]["content"]))
print("usage:", SMOKE.get("usage"))


## 5 · Force an OOM and capture what it says

`control/provisioner/status.py` reports OOM with advice about accelerator size,
GPU count and `--max-model-len`. This checks the advice matches the real error.

Asks for a context far beyond what a 16GB T4 can hold KV cache for.


In [ ]:
import subprocess
proc.terminate(); proc.wait()

oom = subprocess.run(
    ["vllm", "serve", "--model", MODEL, "--port", "8001",
     "--max-model-len", "200000", "--gpu-memory-utilization", "0.95",
     "--dtype", "half"],
    capture_output=True, text=True, timeout=900,
)
OOM_TEXT = (oom.stdout + oom.stderr)
tail = [l for l in OOM_TEXT.splitlines() if l.strip()][-25:]
print("exit", oom.returncode)
print("\n".join(tail))


## 6 · Summary — paste this back


In [ ]:
import json
print(json.dumps({
    "gpu": "T4 (compute 7.5, 16GB)",
    "vllm_version": vllm.__version__,
    "model": MODEL,
    "load_seconds_cold": LOAD_SECONDS,
    "dtype_note": "bfloat16 unsupported on T4; --dtype half required",
    "metric_names": names,
    "keel_depends_on_present": {
        w: (w in names) for w in [
            "vllm:request_success_total", "vllm:num_requests_waiting",
            "vllm:time_to_first_token_seconds", "vllm:time_per_output_token_seconds",
        ]
    },
    "smoke_reply": SMOKE["choices"][0]["message"]["content"],
    "oom_signature": [l for l in OOM_TEXT.splitlines()
                      if any(k in l.lower() for k in
                             ["memory", "kv cache", "reduce", "oom", "valueerror"])][-6:],
}, indent=2))
